Loading '1B' parameter model : "TinyLlama/TinyLlama-1.1B-Chat-v1.0" using transformers

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import time

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    trust_remote_code=True
)

model.eval()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

LlamaForCausalLM(
  (model): LlamaModel(
    (embed_tokens): Embedding(32000, 2048)
    (layers): ModuleList(
      (0-21): 22 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (k_proj): Linear(in_features=2048, out_features=256, bias=False)
          (v_proj): Linear(in_features=2048, out_features=256, bias=False)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (up_proj): Linear(in_features=2048, out_features=5632, bias=False)
          (down_proj): Linear(in_features=5632, out_features=2048, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
        (post_attention_layernorm): LlamaRMSNorm((2048,), eps=1e-05)
      )
    )
    (norm): LlamaRMSNorm((2048,), eps=1e-05)
    (rot

Defining the prompts in the list "prompts" which also provides table and columns

In [4]:
prompts = [
"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :['student'], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: List students with CGPA > 8.0.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :['student'], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: List students with Internships > 2.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :['student'], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Show all students with CGPA < 6.5.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :['student'], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Find students with exactly 3 Internships.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Display students whose CGPA is between 7.0 and 9.0.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Count the total number of students.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Get the average CGPA of students.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Find students with no Internships.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: List students ordered by CGPA descending.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Show top 5 students by CGPA.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: List students with CGPA >= 9.0 and Internships >= 1.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Find students whose student_name starts with 'A'.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Find students whose student_name contains 'Kumar'.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Get the maximum CGPA.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Get the minimum CGPA.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Count students with CGPA above 8.5.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: List students with Internships less than 1.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Show students sorted by Internships ascending.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: Find students with CGPA not equal to 7.0.\n SQL_query: ",

"You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :[student], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: List student_name and CGPA only.\n SQL_query: "
]

In [16]:

def eval_score():
  correct_entities = int(input("'1' if all enities are correct '0' if incorrect or missing"))
  correct_syntax = int(input("'1' if syntax is correct '0' if incorrect"))
  logic = int(input("'1' if logic is correct '0' if incorrect"))
  return correct_entities,correct_syntax,logic
results = []
e = 0
s = 0
l = 0
for prompt in prompts:
    start = time.time()

    inputs = tokenizer(prompt, return_tensors="pt")
    outputs = model.generate(**inputs, max_new_tokens=30)

    latency = time.time() - start

    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(output_text)
    entity,syntax,logic = eval_score()
    #e = e + entity
    #s = s + syntax
    #l = l + logic
    token_count = len(outputs[0])
    results.append({
        "eval_score": [entity,syntax,logic],
        "output": output_text,
        "latency": latency,
        "tokens": token_count
    })

You are a SQL generator which returns only one valid SQL query for the sentence.
 Schema : tables :['student'], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].
Sentence: List students with CGPA > 8.0.
 SQL_query:  SELECT student_id, student_name, CGPA, Internships FROM students WHERE CGPA > 8.0;

Explanation
'1' if all enities are correct '0' if incorrect or missing0
'1' if syntax is correct '0' if incorrect1
'1' if logic is correct '0' if incorrect1
You are a SQL generator which returns only one valid SQL query for the sentence.
 Schema : tables :['student'], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].
Sentence: List students with Internships > 2.
 SQL_query:  SELECT student_id, student_name, CGPA, Internships FROM students WHERE Internships > 2;

Explanation:

'1' if all enities are correct '0' if incorrect or missing0
'1' if syntax is correct '0' if incorrect1
'1' if logic is correct '0' if incorrect1
You are a SQL generato

In [18]:
results[3]['eval_score']

[1, 1, 0]

In [24]:
import re
final_result = []
avg_token = 0
avg_latency = 0
e = 0
s = 0
l = 0
for i in range(len(results)):
  match = re.search(r"SQL_query:\s*(.*?)(?=\n\s*\n|\n\s*[A-Z][a-z]+:|$)",results[i]['output'], re.DOTALL)
  sql_query = match.group(1).strip()

  #print('prompt : ',prompts[i])
  print(sql_query)
  e = e + results[i]['eval_score'][0]
  s = s + results[i]['eval_score'][1]
  l = l + results[i]['eval_score'][2]
  print(f'evaluation : entity {results[i]['eval_score'][0]}, syntax {results[i]['eval_score'][1]}, logic {results[i]['eval_score'][2]}')
  print('\n')
  avg_token = avg_token + results[i]['tokens']
  avg_latency = avg_latency + results[i]['latency']
  final_result.append({'prompt' : prompts[i],'SQL-query':sql_query,'evaluation' : {'entity_score': results[i]['eval_score'][0], 'syntax_score':results[i]['eval_score'][1], 'logic_score':results[i]['eval_score'][2]},'latency' : results[i]['latency'],'tokens' : results[i]['tokens']})

print(f'Final evaluation : entity {results[i]['eval_score'][0]}, syntax {results[i]['eval_score'][1]}, logic {results[i]['eval_score'][2]}')
final_result.append({'average tokens' : avg_token/len(results)})
final_result.append({'average latency' : avg_latency/len(results)})
final_result.append({'Total_eval_score' : f'evaluation : entity {e}, syntax {s}, logic {l}'})
print('average tokens : ',avg_token/len(results))
print('average latency : ',avg_latency/len(results))
print(f'Total_eval_score : evaluation : entity {e}, syntax {s}, logic {l}')
print(final_result)

SELECT student_id, student_name, CGPA, Internships FROM students WHERE CGPA > 8.0;
evaluation : entity 0, syntax 1, logic 1


SELECT student_id, student_name, CGPA, Internships FROM students WHERE Internships > 2;
evaluation : entity 0, syntax 1, logic 1


SELECT * FROM student WHERE CGPA < 6.5;
evaluation : entity 1, syntax 1, logic 1


SELECT student_id, student_name, CGPA, COUNT(INTERNSHIP_ID) FROM student INNER JOIN internships ON student.
evaluation : entity 1, syntax 1, logic 0


SELECT * FROM students WHERE CGPA BETWEEN 7.0 AND 9.0;
evaluation : entity 0, syntax 1, logic 1


SELECT COUNT(*) FROM student;
evaluation : entity 1, syntax 1, logic 1


SELECT AVG(CGPA) FROM student WHERE Internships = 1;
evaluation : entity 0, syntax 1, logic 0


SELECT * FROM students WHERE Internships = 0;
evaluation : entity 0, syntax 1, logic 1


SELECT student_id, student_name, CGPA, Internships FROM students WHERE CGPA > 3.00 ORDER BY CGPA DESC
evaluation : entity 0, syntax 1, logic 0


SELECT *

In [26]:
import json

print(json.dumps(final_result, indent=4))

[
    {
        "prompt": "You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :['student'], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: List students with CGPA > 8.0.\n SQL_query: ",
        "SQL-query": "SELECT student_id, student_name, CGPA, Internships FROM students WHERE CGPA > 8.0;",
        "evaluation": {
            "entity_score": 0,
            "syntax_score": 1,
            "logic_score": 1
        },
        "latency": 28.945062160491943,
        "tokens": 106
    },
    {
        "prompt": "You are a SQL generator which returns only one valid SQL query for the sentence.\n Schema : tables :['student'], columns: [student_id(INT),student_name(CHAR),CGPA(FLOAT),Internships(INT)].\nSentence: List students with Internships > 2.\n SQL_query: ",
        "SQL-query": "SELECT student_id, student_name, CGPA, Internships FROM students WHERE Internships > 2;",
        "evaluation": {
            